# Publish Crux benchmark reports to MLflow (Colab)

Runs `crux benchmark publish-mlflow-cohort` for a set of canonical cohort report
directories stored in Google Drive, then republishes the first one to confirm
idempotency (same run ID, `created=false`). See `docs/benchmark/mlflow.md`.

**Before running**

1. Add a Colab Secret (key icon in the left sidebar) named `MLFLOW_TRACKING_PASSWORD`
   holding your DagsHub access token, with notebook access enabled. The tracking URI
   and username are not secret and are set below.
2. Upload `crux-mlflow-reports.zip` to Drive at `REPORTS_ZIP` below. It holds one folder per
   report (named as in `REPORTS` below), each with `summary.json`, `summary.md`,
   `items.csv`, `per_song.csv`, `per_class.csv`.

Only those five small files per report are uploaded. No GPU is needed.

In [ ]:
CRUX_REF = "agent/hpa-86-mlflow-benchmark-visualization"  # branch or tag to run
REPORTS_ZIP = "/content/drive/MyDrive/crux-hpa86/crux-mlflow-reports.zip"
REPORTS_ROOT = "/content/crux-mlflow-reports"
EXPERIMENT = "crux-benchmark"
MLFLOW_TRACKING_URI = "https://dagshub.com/cwchanap/Crux.mlflow"
MLFLOW_TRACKING_USERNAME = "cwchanap"

# folder name under REPORTS_ROOT -> --scope
REPORTS = {
    "oaf-full-mix-broad": "broad",
    # Add more report folders as they become available, e.g.:
    # "oaf-full-mix-reviewed": "reviewed",
    # "oaf-spleeter-pilot": "pilot",
}

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
!rm -rf "$REPORTS_ROOT" && unzip -q "$REPORTS_ZIP" -d "$REPORTS_ROOT"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_TRACKING_PASSWORD")

missing = [name for name in REPORTS if not os.path.isfile(f"{REPORTS_ROOT}/{name}/summary.json")]
assert not missing, f"missing report folders in {REPORTS_ROOT}: {missing}"

In [ ]:
%cd /content
!rm -rf Crux && git clone -q --depth 1 --branch "$CRUX_REF" https://github.com/cwchanap/Crux.git
%cd /content/Crux
# The publish command only needs these; librosa/numpy are preinstalled on Colab.
!pip install -q "mlflow-skinny>=3.16,<4" "click>=8.2" uuid6

In [ ]:
import json
import subprocess
import sys


def publish(name, scope):
    proc = subprocess.run(
        [
            sys.executable,
            "-m",
            "src.cli.main",
            "benchmark",
            "publish-mlflow-cohort",
            "--reports",
            f"{REPORTS_ROOT}/{name}",
            "--scope",
            scope,
            "--experiment",
            EXPERIMENT,
        ],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"{name}: exit {proc.returncode}\n{proc.stdout}\n{proc.stderr}")
    return json.loads(proc.stdout)


results = {name: publish(name, scope) for name, scope in REPORTS.items()}

first = next(iter(REPORTS))
again = publish(first, REPORTS[first])
assert again["run_id"] == results[first]["run_id"] and again["created"] is False, again
print(f"Republish OK: {first} -> {again['run_id']} ({again['status']})")

In [ ]:
# Paste this table into the PR as hosted smoke evidence (no secrets in it).
print("| Report | Scope | Run ID | Projection SHA-256 | Status |")
print("|---|---|---|---|---|")
for name, r in results.items():
    print(
        f"| {name} | {REPORTS[name]} | `{r['run_id']}` | `{r['projection_sha256'][:12]}` | {r['status']} |"
    )
print(f"\nExperiment: {EXPERIMENT} (`{next(iter(results.values()))['experiment_id']}`)")
print(f"Republish: {first} -> same run, created=false")